In [1]:
!pip install pydantic_settings


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

In [3]:
from functools import lru_cache

from langgraph.graph import END, START, StateGraph

from src.nodes import (
    context_injection_node,
    conversation_node,
)
from src.state import AICompanionState


@lru_cache(maxsize=1)
def create_workflow_graph():
    graph_builder = StateGraph(AICompanionState)

    # Add all nodes
    graph_builder.add_node("context_injection_node", context_injection_node)
    graph_builder.add_node("conversation_node", conversation_node)

    # Define the flow
    graph_builder.add_edge(START, "context_injection_node")
    graph_builder.add_edge("context_injection_node", "conversation_node")
    graph_builder.add_edge("conversation_node", END)

    return graph_builder


# Compiled without a checkpointer. Used for LangGraph Studio
graph = create_workflow_graph().compile()

In [4]:
from langchain_core.messages import HumanMessage, AIMessage


messages = [
    HumanMessage(content="Heyyy! What are you doing?"),
]

# Single turn test
state = {
    "messages": [HumanMessage(content="Hey Jarvis, what are you up to right now?")],
    "summary": "",
    "current_activity": "",
    "apply_activity": False,
}

result = await graph.ainvoke(state)
result


{'messages': [HumanMessage(content='Hey Jarvis, what are you up to right now?', additional_kwargs={}, response_metadata={}, id='c67f38ef-5355-47a7-a456-0fbc95a05972'),
  AIMessage(content="Hey! Quick question first—what's your name? Also, right now I'm heads-down on Groq's ML infra improvements and model performance, and maybe plotting ramen later.", additional_kwargs={}, response_metadata={}, id='e6fede77-9780-44eb-8516-272351bac73d', tool_calls=[], invalid_tool_calls=[])],
 'summary': '',
 'current_activity': "Focused work on improving Groq's ML infrastructure and model performance.",
 'apply_activity': True}

In [ ]:
!pip install gradio

  Using cached gradio-6.6.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached brotli-1.2.0-cp312-cp312-macosx_10_13_x86_64.whl.metadata (6.1 kB)
  Using cached fastapi-0.129.0-py3-none-any.whl.metadata (30 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.1.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-macosx_10_13_x86_64.whl.metadata (2.7 kB)
  Using cached numpy-2.4.2-cp312-cp312-macosx_14_0_x86_64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.1-cp312-cp312-macosx_10_13_x86_64.whl.metadata (79 kB)
  Using cached pillow-12.1.1-cp312-cp312-macosx_10_13_x86_64.whl.metadata (8.8 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)


In [5]:
import gradio as gr
from langchain_core.messages import HumanMessage

conversation_state = {
    "messages": [],
    "summary": "",
    "current_activity": "",
    "apply_activity": False,
}

async def chat(message, history):
    conversation_state["messages"].append(HumanMessage(content=message))
    result = await graph.ainvoke(conversation_state)
    conversation_state.update(result)
    return result["messages"][-1].content

demo = gr.ChatInterface(fn=chat, title="Jarvis")
demo.launch(inline=True, quiet=True)